In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, pickle
ROOT = '/content/drive/MyDrive/CW_Folder_UG'
sys.path.insert(0, f'{ROOT}/Code')
from utils import *
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
import joblib

MODELS_DIR = f'{ROOT}/Models'
os.makedirs(MODELS_DIR, exist_ok=True)

with open(f'{ROOT}/Code/data_splits.pkl', 'rb') as f:
    d = pickle.load(f)
train_paths, train_labels = d['train_paths'], d['train_labels']
test_paths,  test_labels  = d['test_paths'],  d['test_labels']
y_test = np.array(test_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import shutil, os

LOCAL_ROOT = '/tmp/CW_data'

if not os.path.exists(LOCAL_ROOT):
    print("Copying dataset to local SSD.")
    shutil.copytree(ROOT, LOCAL_ROOT)
    print("Done.")
else:
    print("Local copy already exists, skipping.")

# Remap paths from Drive to local SSD
def remap(paths, src=ROOT, dst=LOCAL_ROOT):
    return [p.replace(src, dst) for p in paths]

train_paths = remap(train_paths)
test_paths  = remap(test_paths)
print(f"Paths remapped to {LOCAL_ROOT}")

In [ ]:
tr_idx, val_idx = train_test_split(range(len(train_paths)), test_size=0.1,
                                   stratify=train_labels, random_state=SEED)
tr_p  = [train_paths[i] for i in tr_idx]; tr_l  = [train_labels[i] for i in tr_idx]
val_p = [train_paths[i] for i in val_idx]; val_l = [train_labels[i] for i in val_idx]

cw_tensor = torch.tensor(
    [get_class_weights(tr_l)[i] for i in range(4)], dtype=torch.float
).to(device)

def make_loaders_224(augment=False, bs=32):
    """224×224 loaders. Smaller batch size due to GPU memory requirements."""
    tr_ds  = AgeDataset(tr_p,  tr_l,  get_transforms(CNN_SIZE, augment=augment))
    val_ds = AgeDataset(val_p, val_l, get_transforms(CNN_SIZE))
    te_ds  = AgeDataset(test_paths, test_labels, get_transforms(CNN_SIZE))
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr_ds,  bs, shuffle=True,  **kw),
            DataLoader(val_ds, bs, shuffle=False, **kw),
            DataLoader(te_ds,  bs, shuffle=False, **kw))

print(f'Train: {len(tr_p)}  Val: {len(val_p)}  Test: {len(test_paths)}')

In [ ]:
def build_vgg16_frozen():
    """VGG-16: backbone frozen, only new FC head trained."""
    m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for p in m.features.parameters():
        p.requires_grad = False
    m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    return m.to(device)

def build_vgg16_finetune():
    """VGG-16: all layers unfrozen for full fine-tuning at low LR."""
    m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    return m.to(device)

def build_resnet50_frozen():
    """ResNet-50: backbone frozen, only final FC retrained."""
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(2048, NUM_CLASSES)
    return m.to(device)

def build_resnet50_finetune():
    """ResNet-50: all layers unfrozen for full fine-tuning."""
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(2048, NUM_CLASSES)
    return m.to(device)

In [ ]:
def run_transfer(name, model, lr_head, lr_backbone=None, augment=True, epochs=20):
    """Train a transfer learning model. """
    save_path = f'{MODELS_DIR}/track4_{name}.pt'
    tr_dl, val_dl, te_dl = make_loaders_224(augment=augment)
    criterion = nn.CrossEntropyLoss(weight=cw_tensor)

    if lr_backbone is not None:
        head_params     = [p for n_, p in model.named_parameters()
                           if 'fc' in n_ or 'classifier.6' in n_]
        backbone_params = [p for n_, p in model.named_parameters()
                           if 'fc' not in n_ and 'classifier.6' not in n_]
        opt = optim.Adam(
            [{'params': backbone_params, 'lr': lr_backbone},
             {'params': head_params,     'lr': lr_head}],
            weight_decay=1e-4
        )
    else:
        opt = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_head, weight_decay=1e-4
        )

    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    print(f'Training: {name}')
    t0 = time.time()
    history = train_model(model, tr_dl, val_dl, criterion, opt,
                          sched, epochs, device, save_path)
    tt = time.time() - t0
    val_acc = max(history['val_acc'])
    model.load_state_dict(torch.load(save_path))
    _, preds = eval_epoch(model, te_dl, device)
    test_acc = accuracy_score(y_test, preds)
    plot_history(history, title=name)
    return {'val_acc': val_acc, 'test_acc': test_acc, 'train_time': tt,
            'test_preds': preds, 'path': save_path, 'model': model}

In [ ]:
res_4a = run_transfer('VGG16_frozen', build_vgg16_frozen(),
                      lr_head=1e-3, augment=True, epochs=20)

In [ ]:
res_4b = run_transfer('VGG16_finetune', build_vgg16_finetune(),
                      lr_head=1e-4, lr_backbone=1e-5, augment=True, epochs=20)

In [ ]:
res_4c = run_transfer('ResNet50_frozen', build_resnet50_frozen(),
                      lr_head=1e-3, augment=True, epochs=20)

In [ ]:
res_4d = run_transfer('ResNet50_finetune', build_resnet50_finetune(),
                      lr_head=1e-4, lr_backbone=1e-5, augment=True, epochs=20)

In [ ]:
print('\n' + '='*55)
print('Training: ResNet50 features + RBF SVM')
print('='*55)

resnet_feat = build_resnet50_frozen()
resnet_feat.fc = nn.Identity()   # remove FC to get 2048-d embeddings
resnet_feat.eval()

@torch.no_grad()
def extract_resnet_features(paths, bs=64):
    ds = AgeDataset(paths, [0]*len(paths), get_transforms(CNN_SIZE))
    dl = DataLoader(ds, bs, shuffle=False, num_workers=2)
    feats = []
    for imgs, _ in dl:
        feats.append(resnet_feat(imgs.to(device)).cpu().numpy())
    return np.vstack(feats)

print('Extracting ResNet features (train)...')
X_rn_train = extract_resnet_features(train_paths)
print('Extracting ResNet features (test)...')
X_rn_test  = extract_resnet_features(test_paths)

(X_rn_tr, X_rn_val, y_tr, y_val) = train_test_split(
    X_rn_train, np.array(train_labels),
    test_size=0.1, stratify=train_labels, random_state=SEED
)
X_rn_tr_s, X_rn_val_s, X_rn_te_s, sc_rn = fit_scaler(X_rn_tr, X_rn_val, X_rn_test)

print('Fitting RBF SVM on ResNet-50 features...')
t0 = time.time()
svm_rn = svm.SVC(C=10, kernel='rbf', gamma='scale',
                 class_weight='balanced', random_state=SEED)
svm_rn.fit(X_rn_tr_s, y_tr)
tt = time.time() - t0
preds_4e     = svm_rn.predict(X_rn_te_s)
val_preds_4e = svm_rn.predict(X_rn_val_s)
joblib.dump(svm_rn, f'{MODELS_DIR}/track4_ResNet50_SVM.pkl')
res_4e = {'val_acc':  accuracy_score(y_val,  val_preds_4e),
          'test_acc': accuracy_score(y_test, preds_4e),
          'train_time': tt, 'test_preds': preds_4e,
          'path': f'{MODELS_DIR}/track4_ResNet50_SVM.pkl'}
print(f"  val={res_4e['val_acc']:.4f}  test={res_4e['test_acc']:.4f}  time={tt:.1f}s")

In [ ]:
results = {
    'VGG16_frozen':      res_4a,
    'VGG16_finetune':    res_4b,
    'ResNet50_frozen':   res_4c,
    'ResNet50_finetune': res_4d,
    'ResNet50_feat+SVM': res_4e,
}
print(f"{'Model':<25} {'Val Acc':>9} {'Test Acc':>10} {'Time(s)':>9}")
print('─' * 57)
for n, r in sorted(results.items(), key=lambda x: -x[1]['test_acc']):
    print(f"{n:<25} {r['val_acc']:>9.4f} {r['test_acc']:>10.4f} {r['train_time']:>9.1f}")

best_name = max(results, key=lambda n: results[n]['test_acc'])
print(f"\nBest Track 4: {best_name}  (test acc={results[best_name]['test_acc']:.4f})")

In [ ]:
for name, r in results.items():
    evaluate(name, y_test, r['test_preds'], r['train_time'], r['path'])

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5))
for ax, (n, r) in zip(axes, results.items()):
    plot_confusion_matrix(n, y_test, r['test_preds'], ax=ax)
plt.suptitle('Track 4 Transfer Learning Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
names  = list(results.keys())
accs   = [results[n]['test_acc'] for n in names]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(names, accs, color=colors)
ax.set_ylim(0, 1); ax.set_ylabel('Test Accuracy')
ax.set_title('Track 4 Transfer Learning Model Comparison')
plt.xticks(rotation=20, ha='right')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005, f'{acc:.3f}',
            ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
qualitative_grid(test_paths, test_labels,
                 {n: r['test_preds'] for n, r in results.items()}, n=8)

In [ ]:
results_save = {n: {k: v for k, v in r.items() if k != 'model'}
                for n, r in results.items()}
with open(f'{ROOT}/Code/track4_results.pkl', 'wb') as f:
    pickle.dump({'results': results_save, 'best_name': best_name}, f)
print('Track 4 results saved')